# HTO Interactive Image Gallery
This notebook provides an interactive gallery of X-ray images with overlays from two different sources:
1. **Specialists Labels** (from `specialists_labels.json`)
2. **HTO Annotations** (from `hto_annotations.json`)


In [1]:
import os
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline

# Configuration
DATA_DIR = "../data/hto/xrays"
HTO_ANN_PATH = os.path.join(DATA_DIR, "hto_annotations.json")
SPEC_LABELS_PATH = os.path.join(DATA_DIR, "specialists_labels.json")

# Load data
print("Loading annotations...")
with open(HTO_ANN_PATH, 'r') as f:
    hto_data = json.load(f)

with open(SPEC_LABELS_PATH, 'r') as f:
    spec_data = json.load(f)

# Prepare image mapping
image_id_to_file = {img['id']: img['file_name'] for img in hto_data['images']}
file_to_image_id = {img['file_name']: img['id'] for img in hto_data['images']}
image_list = sorted(list(file_to_image_id.keys()))

# Prepare annotations mapping
hto_annotations_by_image = {}
for ann in hto_data['annotations']:
    img_id = ann['image_id']
    if img_id not in hto_annotations_by_image:
        hto_annotations_by_image[img_id] = []
    hto_annotations_by_image[img_id].append(ann)

print(f"Loaded {len(image_list)} images.")


Loading annotations...
Loaded 70 images.


In [2]:
def draw_hto_overlay(ax, img_name):
    """Draws keypoints from hto_annotations.json"""
    img_id = file_to_image_id.get(img_name)
    if img_id is None: return
    
    anns = hto_annotations_by_image.get(img_id, [])
    for ann in anns:
        kp = ann.get('keypoints', [])
        for i in range(0, len(kp), 3):
            x, y, v = kp[i], kp[i+1], kp[i+2]
            if v > 0:
                ax.plot(x, y, 'ro', markersize=4)

def draw_spec_overlay(ax, img_name):
    """Draws keypoints from specialists_labels.json"""
    colors = ['blue', 'green', 'magenta', 'orange', 'cyan']
    specialists = list(spec_data.keys())
    
    for i, spec_key in enumerate(specialists):
        spec_images = spec_data[spec_key]
        if img_name in spec_images:
            landmarks = spec_images[img_name]
            color = colors[i % len(colors)]
            spec_name = spec_key.split('/')[-1]
            
            first_point = True
            for side in ['left', 'right']:
                if side in landmarks:
                    for lm_name, coords in landmarks[side].items():
                        label = spec_name if first_point else None
                        ax.plot(coords['x'], coords['y'], 'o', color=color, markersize=4, label=label)
                        first_point = False
    if specialists:
        # Remove duplicate labels in legend
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        if by_label:
            ax.legend(by_label.values(), by_label.keys(), loc='upper right', fontsize='small')


In [3]:
output_widget = widgets.Output()

def update_display(change=None):
    index = index_slider.value
    img_name = image_list[index]
    img_path = os.path.join(DATA_DIR, img_name)
    
    with output_widget:
        clear_output(wait=True)
        if not os.path.exists(img_path):
            print(f"Image {img_name} not found at {img_path}")
            return
            
        try:
            img = Image.open(img_path)
        except Exception as e:
            print(f"Error opening image {img_path}: {e}")
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 12))
        
        # Specialists side
        ax1.imshow(img, cmap='gray')
        draw_spec_overlay(ax1, img_name)
        ax1.set_title(f"Specialists Labels: {img_name}")
        ax1.axis('off')
        
        # HTO side
        ax2.imshow(img, cmap='gray')
        draw_hto_overlay(ax2, img_name)
        ax2.set_title(f"HTO Annotations: {img_name}")
        ax2.axis('off')
        
        plt.tight_layout()
        plt.show()

# Widgets
index_slider = widgets.IntSlider(
    value=0, min=0, max=len(image_list)-1, 
    description='Image:', layout=widgets.Layout(width='60%')
)
prev_button = widgets.Button(description='◀ Previous', button_style='info')
next_button = widgets.Button(description='Next ▶', button_style='info')

def on_prev_clicked(b):
    if index_slider.value > 0:
        index_slider.value -= 1

def on_next_clicked(b):
    if index_slider.value < len(image_list) - 1:
        index_slider.value += 1

prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)
index_slider.observe(update_display, names='value')

# Display UI
controls = widgets.HBox([prev_button, index_slider, next_button])
display(controls)
display(output_widget)

# Initial call
update_display()


Output()